# CSER Colab Ablation Runner

Use this notebook to run the automated SER ablation framework on Google Colab. Upload a zipped project folder to Google Drive first, then run the cells below.

In [ ]:
# 1. Check GPU
!nvidia-smi || true

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Configure paths
# Upload CSER.zip to Google Drive first. Recommended path:
# /content/drive/MyDrive/CSER_slim.zip

PROJECT_ZIP = '/content/drive/MyDrive/CSER_slim.zip'
PROJECT_DIR = '/content/CSER'

# Optional: set this to a Drive folder if you want outputs copied back after each run.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/CSER_colab_outputs'

import os, shutil, subprocess, json, textwrap
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
print('PROJECT_ZIP:', PROJECT_ZIP)
print('PROJECT_DIR:', PROJECT_DIR)
print('DRIVE_OUTPUT_DIR:', DRIVE_OUTPUT_DIR)

In [ ]:
# 3. Unzip project
import os, shutil, zipfile
from pathlib import Path

if not os.path.exists(PROJECT_ZIP):
    raise FileNotFoundError(f'Upload project zip first: {PROJECT_ZIP}')

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

# Clean files incorrectly extracted from Windows zip paths, e.g. /content/CSER\\configs\\*.yaml
for p in Path('/content').iterdir():
    if p.name.startswith('CSER\\'):
        p.unlink()

with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    for info in zf.infolist():
        name = info.filename.replace('\\', '/').lstrip('/')
        if not name or name.endswith('/'):
            continue
        parts = [part for part in name.split('/') if part not in ('', '.', '..')]
        if not parts:
            continue
        target = Path('/content').joinpath(*parts)
        if not str(target.resolve()).startswith('/content/'):
            raise RuntimeError(f'Unsafe zip path: {info.filename}')
        target.parent.mkdir(parents=True, exist_ok=True)
        with zf.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)

# If the zip contains a differently named root folder, locate it.
if not os.path.exists(PROJECT_DIR):
    candidates = [p for p in os.listdir('/content') if os.path.isdir('/content/' + p) and p.lower().startswith('cser')]
    if not candidates:
        raise RuntimeError('Could not find extracted CSER folder under /content')
    os.rename('/content/' + candidates[0], PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls -la

In [ ]:
# 4. Install dependencies
# Colab already provides torch; keep the install list focused to avoid replacing CUDA wheels.
!pip -q install pandas numpy pyarrow tqdm scikit-learn matplotlib seaborn librosa soundfile thop pyyaml

In [ ]:
# 5. Verify data and scripts
import os

required = [
    'dataset/IEMOCAP/final_data/train_v3.parquet',
    'dataset/IEMOCAP/final_data/test_v3.parquet',
    'experiments/run_experiments.py',
    'experiments/train_worker.py',
]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    print('Missing files:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Project package is incomplete. Include dataset/IEMOCAP/final_data/*.parquet in the zip or copy data into PROJECT_DIR.')

!python -m py_compile experiments/run_experiments.py experiments/run_efficiency.py experiments/run_noise_eval.py experiments/summarize_results.py experiments/train_worker.py
print('Verification passed.')

In [ ]:
# 6. Smoke test: 1 experiment, 1 epoch, 1 batch
# This checks that Colab can load data, train, save checkpoint, and write results.
!python experiments/run_experiments.py --config experiments/configs/lightweight.yaml --seeds 42 --max-experiments 1 --epochs 1 --max-batches 1 --skip-followups

In [ ]:
# 7. Recommended staged runs
# Run these one by one. They are safer than starting --all immediately.

# Lightweight comparison: Tiny / Full / Large / Gated Full
# !python experiments/run_experiments.py --config experiments/configs/lightweight.yaml

# Core comparison, first with one seed to estimate runtime
# !python experiments/run_experiments.py --config experiments/configs/core_models.yaml --seeds 42

# Module ablations
# !python experiments/run_experiments.py --config experiments/configs/ablations.yaml

print('Uncomment the staged command you want to run.')

In [ ]:
# 8. Full automatic run
# Warning: this may take days depending on Colab GPU allocation.
# Uncomment only after the smoke test succeeds.

# !python experiments/run_experiments.py --all

print('Full run cell is intentionally commented for safety.')

In [ ]:
# 9. If you ran training with --skip-followups, run evaluations manually.
# Efficiency and noise evaluation read experiments/output/results/*.json.

!python experiments/run_efficiency.py --results experiments/output/results
!python experiments/run_noise_eval.py --results experiments/output/results --noise-types white babble --snr 20 10 5 0
!python experiments/summarize_results.py

In [ ]:
# 10. Copy outputs back to Google Drive
import shutil, os, time

stamp = time.strftime('%Y%m%d_%H%M%S')
archive_base = f'/content/CSER_experiment_outputs_{stamp}'
archive_path = shutil.make_archive(archive_base, 'zip', 'experiments/output')
target = os.path.join(DRIVE_OUTPUT_DIR, os.path.basename(archive_path))
shutil.copy2(archive_path, target)
print('Saved output archive to:', target)